<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 1: </b>코스 환경</h2>
<br>

이 모듈에서는 코스 환경을 소개하며, 환경 구성 요구 사항과 워크플로, 그리고 고려해야 할 점들을 살펴봅니다.

**참고:** 이 노트북은 **Google Colab**에서도 열 수 있지만, 모든 셀을 제대로 실행하려면 ***DLI 코스 환경***이 필요합니다. 다만 이 섹션에는 직접 작성해야 할 TODO가 없고, 주로 뒤에서 어떤 일이 일어나는지 이해하기 위한 용도이므로 그냥 읽기만 해도 학습 경험에는 큰 지장이 없습니다. 아래의 호스트 명령 출력은 개략적인 예시이며, 실제 서비스 이름과 포트는 코스 배포 방식에 따라 달라집니다.

**권장 사항:** 코스 환경을 열어 잠시 익숙해지는 것이 좋지만 필수는 아닙니다. 나중에 해도 괜찮습니다. **환경을 사용하지 않을 때는 세션을 종료해 두는 것을 권장합니다!**

<br>

### **학습 목표:**

- 코스 환경이 어떻게 만들어졌고 왜 이런 식으로 구성되었는지의 관점에서 환경을 이해합니다.
- Jupyter Labs 인터페이스를 사용해 활성 네트워크 포트를 통해 주변 마이크로서비스와 상호작용하는 방법을 이해합니다.

<br>

### **생각해 볼 질문:**

1. 이 코스의 환경에는 어떤 종류의 리소스가 있을 것으로 예상되며, 여러분의 로컬 컴퓨팅 환경과는 어떻게 다를까요?
2. 마이크로서비스 중 하나가 다른 호스트 환경(공개 접근 가능 또는 접근이 제한된)에서 실행된다면 상황이 얼마나 달라질까요?
    - **같은 아이디어, 다른 질문**: 원격 호스트에서 제공되는 서비스를 마치 로컬 마이크로서비스처럼 동작하도록 흉내 내는 것은 얼마나 어려울까요? 그리고 그렇게 했을 때 본질적인 단점이 있을까요?
3. 실제로 사용자별로 띄워야 하는 마이크로서비스는 어떤 것이고, 계속 실행 상태로 두는 것이 나은 서비스는 어떤 것일까요?

<br>

<br>

## **클라우드 환경에 오신 것을 환영합니다**

이곳은 코스 콘텐츠를 진행하는 데 사용할 수 있는 Jupyter Labs 환경입니다. 대부분의 코스에서 이 환경은 필요한 구성 요소가 이미 백그라운드에서 실행되고 있는 주어진 인터페이스일 뿐입니다. 하지만 이 코스에서는 더 깊은 탐구를 유도하기 위해, 이 환경을 마이크로서비스 오케스트레이션(특히 **대규모 언어 모델(LLM)** 중심 애플리케이션에서의)을 이해하는 관문으로도 활용합니다. 먼저 클라우드 세션의 핵심 구성 요소부터 살펴보겠습니다.

<!-- <img src="https://drive.google.com/uc?export=view&id=11MGA5fkwA1XQAglQYQbOgjGTImO3TkLS" width=800/> -->
<!-- <img src="imgs/simple-env.png" width=800/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/simple-env.png" width=800px/>

----

<br>

## **Part 1:** 컨테이너 호스팅

이 Jupyter Notebook에 접속하면 NVIDIA DLI(Deep Learning Institute)가 AWS나 Azure 같은 클라우드 플랫폼의 인스턴스 하나를 여러분에게 할당합니다. 이 클라우드 인스턴스가 기본 클라우드 환경이 되며, 다음을 포함합니다:

- 처리를 위한 전용 CPU, 그리고 경우에 따라 GPU.
- 미리 설치된 기본 운영체제.
- 알려진 웹 주소를 통해 접근할 수 있도록 노출된 몇 개의 포트.

이것만으로도 시작에 필요한 리소스는 모두 갖춰지지만, 기본 상태에서는 사실상 빈 도화지에 불과합니다. 원한다면 몇 가지 정해진 루틴을 실행해 리소스를 다운로드하고 환경 전체를 완전히 개방된 상태로 노출할 수도 있습니다. 하지만 백그라운드에서 다른 프로세스가 실행되어야 하는 상황이라면 이는 좋은 생각이 아닐 수 있습니다. 예를 들어 데이터베이스 서비스를 띄우거나, 큰 문서를 로드하거나, 안전한 연결을 위한 프록시 서비스를 구성하고 싶을 수도 있습니다.

기본 구성을 다양한 프로세스를 갖춘 실질적인 개발 공간으로 바꾸기 위해, 사용자나 시스템이 의존할 수 있는 일련의 마이크로서비스를 백그라운드에 배포해 두었습니다. [**마이크로서비스(Microservices)**](https://en.wikipedia.org/wiki/Microservices)는 특정 기능을 수행하며 가벼운 연결 프로토콜로 서로 통신하는 자율적인 서비스입니다. 여러분의 환경에는 Jupyter Labs 서버와 함께, 조사하고 실험해 볼 만한 여러 다른 서비스가 포함되어 있습니다.

<br>

<!-- <img src="https://drive.google.com/uc?export=view&id=1r0BH_zROmGosrsUt_hhAY4azXc4wtjea" width=800/> -->
<!-- <img src="imgs/docker-ms.png" width=1000/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/docker-ms.png" width=1000px/>

마이크로서비스 오케스트레이션에 [**Docker**](https://www.docker.com/)를 활용하면, **컨테이너화(containerization)** 와 **일관성(uniformity)** 같은 원칙을 따르는 새로운 마이크로서비스를 비교적 간단하게 추가할 수 있습니다:

- **컨테이너화:** 각 서비스를 코드, 런타임, 라이브러리, 시스템 도구 등 필요한 소프트웨어 구성 요소를 모두 포함하는 독립적인 컨테이너로 감싸는 과정입니다. 이 컨테이너들은 네트워크 포트를 통해 호스트 리소스 및 다른 서비스와 상호작용합니다. 주요 장점은 다음과 같습니다:
    - **이식성(Portability):** 다양한 환경으로 쉽게 이전하고 배포할 수 있습니다.
    - **격리(Isolation):** 각 컨테이너가 독립적으로 동작하도록 보장하여 서비스 간 충돌을 최소화합니다.
    - **확장성(Scalability):** 변화하는 수요에 맞춰 서비스를 확장하거나 *배포 토폴로지*(어떤 서비스가 어떤 리소스 위에서 실행되는지, 어디에 위치하는지, 누가 접근하는지)를 변경하는 과정을 단순화합니다.

- **일관성:** Docker는 서로 다른 환경에서도 일관되게 동작하는 것을 목표로 하여 각 마이크로서비스가 안정적으로 수행되도록 합니다. 다만 몇 가지 제약은 알아 둘 필요가 있습니다:
    - **하드웨어 민감성:** 하드웨어가 다른 환경에서는 성능이 달라질 수 있으므로, 적응 가능한 마이크로서비스 설계가 필요합니다.
    - **환경적 요인:** 네트워크 지연이나 저장 용량 같은 변수가 컨테이너 효율에 영향을 줄 수 있습니다.

마이크로서비스 오케스트레이션을 위한 Docker와 컨테이너화에 대한 더 포괄적인 내용은 시간이 될 때 [Docker 공식 문서](https://docs.docker.com/)를 참고하시길 권장합니다. 이러한 원칙을 이해해 두면 실제 배포로 나아가고자 하는 분들에게 매우 유용할 것입니다.

----

<br>

## **Part 2:** Jupyter Labs 마이크로서비스

일반적인 마이크로서비스에 대해 이야기했으니, 이제 여러분이 지금까지 계속 사용해 온 **Jupyter Labs 마이크로서비스**에 집중해 보겠습니다. 이 대화형 웹 애플리케이션을 통해 원격 호스트에 설치된 소프트웨어로 Python 코드를 작성하고 실행할 수 있습니다(그 외에도 많은 일을 할 수 있습니다)! [Google Colab](https://colab.research.google.com/?utm_source=scs-index) 같은 웹 기반 서비스로 이미 익숙하시겠지만, 이 환경이 *왜* 존재하고 뒤에서 어떻게 동작하는지는 생각해 본 적이 없을지도 모릅니다. LLM 애플리케이션을 위한 마이크로서비스 오케스트레이션을 다루는 만큼, 오늘은 이를 살펴보기에 좋은 날입니다!

**질문:** 왜 코스 환경에 Jupyter Notebook이 있을까요?

**답변:** 마운트된 `composer/deploy/docker-compose.yml`에는 컨테이너 이름이 `jupyter-notebook-server`인 서비스가 정의되어 있습니다. 배포 또는 빌드 오버라이드에 따라 `build`나 `image` 설정이 제공될 수 있으며, 핵심 프로파일은 다음과 같습니다:

```yaml
  lab:
    container_name: jupyter-notebook-server
    build:
      context: ..
      dockerfile: composer/Dockerfile
    ports: # Maps a port on the host to a port in the container.
    - "9010:9010"
    - "9011:9011"
    - "9012:9012"
```

한 문장으로 요약하면, 이 구성 요소는 [`composer/Dockerfile`](./composer/Dockerfile)의 루틴을 파일 상단에 지정된 이미지(`python`이 미리 설치된 슬림 이미지임을 알 수 있습니다)로부터 실행하여 `jupyter-notebook-server`라는 컨테이너 이름의 서비스를 만듭니다.

이 빌드가 끝나고 실행 시 오류가 없으면, 사용자는 실행 중인 Jupyter Labs 세션에 접속해 제공된 인터페이스를 사용할 수 있습니다!

----

<br>

## **Part 3:** 호스트 입장에서 마이크로서비스와 상호작용하기

Jupyter를 제공하는 이 마이크로서비스가 존재하고 지금 우리가 그것과 상호작용하고 있다는 것을 확인했습니다. 그럼... 다른 것들은 무엇이 있을까요? 앞서 언급한 마운트된 `composer/deploy/docker-compose.yml`을 살펴보면 시작 루틴의 일부로 어떤 다른 구성 요소가 만들어졌는지 확인할 수 있습니다. 이 파일의 한 버전이 (마이크로서비스 바깥의) 호스트 환경에서 다음과 같은 명령으로 실행되기 때문입니다:

```text
> docker compose up -d
## Schematic output; exact services come from composer/deploy/docker-compose.yml
[+] Running
 ✔ Container jupyter-notebook-server  Started
 ✔ Container llm_client               Started
 ✔ Container frontend                 Started
 ... optional services vary by locale and deployment overrides
```

### **Jupyter Labs 마이크로서비스 *바깥*에서 상호작용하기**

마이크로서비스가 시작된 뒤에는 호스트 환경에서 간단한 명령 `docker ps -a`(또는 더 간결한 버전)로 다른 마이크로서비스의 상태를 확인해 볼 수 있습니다:

In [ ]:
'''
Schematic output; inspect composer/deploy/docker-compose.yml for exact names and ports.
> docker ps --format "table {{.Names}}\t{{.Image}}\t{{.Ports}}"
NAMES                     IMAGE     PORTS
jupyter-notebook-server   ...       ...
llm_client                ...       ...
frontend                  ...       ...
... optional services vary by locale and deployment overrides
'''

호스트에서 이 명령을 실행하면 실행 중인 컨테이너 목록이 나오며, 컨테이너 바깥에서 마이크로서비스와 상호작용하기 위한 출발점이 됩니다. 이 맥락에서 할 수 있는 일들은 다음과 같습니다:

- `scp`(secure copy protocol)나 `docker cp` 같은 루틴으로 컨테이너와 파일을 주고받기.
    - `docker cp jupyter-notebook-server:/dli/task/paint-cat.jpg .`
- 실행 중인 컨테이너에서 명령 실행하기.
    - `docker exec -it jupyter-notebook-server /bin/bash -c "ls"`
- 컨테이너의 로그를 조회하여 상태와 실행 과정 확인하기.
    - `docker logs jupyter-notebook-server`

<br>

### **Jupyter Labs 마이크로서비스 *안*에서 상호작용하기**

컨테이너 안에서는 노출된 포트와 제공된 리소스를 통해서만 다른 컨테이너와 통신할 수 있습니다. 예를 들어, 이 Jupyter Labs 노트북에는 Docker가 설치되어 있지도 않으며, 호스트의 Docker 인스턴스에 접근할 수도 없습니다:

In [ ]:
## Should fail
!docker ps -a

<br>

이는 보안 측면에서는 대체로 바람직하지만, 다른 마이크로서비스와 상호작용하기는 까다로워질 수 있습니다. 그렇다면 컨테이너 안에서 정확히 무엇을 할 수 있을까요?

호스트 환경에서는 `docker_router` 서비스 같은 것을 통해 바깥 세계로 향하는 아주 작은 창을 제공할 수 있습니다. 이 서비스를 만드는 데 사용된 코드는 [`docker_router/docker_router.py`](docker_router/docker_router.py)와 [`docker_router/Dockerfile`](docker_router/Dockerfile)에서 확인할 수 있으며, 이를 보면 `help`가 조회 가능한 항목 중 하나임을 바로 알 수 있습니다. 아래는 `help` 루틴을 호출하는 데 사용할 수 있는 셸 네트워크 조회 명령의 예시입니다:

In [ ]:
## Should fail in colab, will work in course environment
!curl -v docker_router:8070/help

<br>

위에서 보인 `curl` 인터페이스는 일반적으로 매우 유용하지만 Python 환경에는 다소 최적화되어 있지 않습니다. 다행히 Python의 `requests` 라이브러리가 훨씬 다루기 쉬운 유틸리티를 제공하므로, 앞서 힌트를 얻은 containers 경로를 다음과 같이 더 Python다운 인터페이스로 조회해 보겠습니다:

In [ ]:
## Should fail in colab, will work in course environment
import requests

## Curl request. Best for shell environments
# !curl -v docker_router:8070/containers

## Print all running containers
# requests.get("http://docker_router:8070/containers", timeout=10).json()

## Print the running microservices
for entry in requests.get("http://docker_router:8070/containers", timeout=10).json():
    if entry.get("status") == 'running':
        print(entry.get("name"))

<br>

이를 통해 최소한 어떤 마이크로서비스가 실행 중인지 알 수 있고, 각각의 용도가 무엇일지 생각해 볼 수 있습니다:
- **docker_router**: 이 정보를 얻기 위해 상호작용하고 있는 서비스.
- **jupyter-notebook-server**: 앞서 이야기한, 이 Jupyter 세션을 실행하는 서비스.
- **frontend**: 아마도 일종의 웹 인터페이스...
- **llm_client**: 아마도 일종의 LLM 서버?
- **s-fx-<...>**: 몇 가지 백그라운드 서비스(데이터 로더, 프록시 서비스, 평가 관리자)로, 따로 다루지는 않습니다.

<!-- <img src="imgs/environment.png" width=800/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/environment.png" width=800px/>


마지막 몇 개 구성 요소를 제외하면, 이 구성 요소들에 관한 모든 세부 사항은 다시 한번 [`composer`](composer) 디렉터리에서 찾아볼 수 있습니다.

----

<br>

## **Part 4:** 프론트엔드 확인하기

무엇보다도 이 노트북은 환경을 탐색할 수 있도록 열어 두고, 마이크로서비스 구성 세부 사항에 관심이 있다면 살펴볼 만한 방향을 제시하기 위한 것입니다. 코스 전반에 걸쳐 이 마이크로서비스들 중 일부와 상호작용하게 될 수 있으므로, 어떻게 만들어졌는지 알아 두면 분명 유용할 것입니다!

이왕 하는 김에, 우리가 상호작용해야 할 또 다른 주요 마이크로서비스인 **프론트엔드**도 살펴보겠습니다. 이 마이크로서비스는 최종 평가에서 사용해야 할 웹페이지를 호스팅합니다. 아래 curl 명령을 실행하여 프론트엔드 서비스가 정상 실행 중인지 확인해 주세요! 

In [ ]:
## Commented out by default since it will yield a lot of output
# !curl -v frontend:8090

이 명령은 `200 OK` 응답과 함께 웹페이지(즉, `<!doctype html>`로 시작하는 응답)를 반환해야 합니다. 이는 유용한 헬스 체크이긴 하지만 사용자 친화적이지는 않습니다. 웹페이지에 접근하려면:

- **원시 포트 접근 (기본값)**: 브라우저에 `http://<...>.courses.nvidia.com:8090`을 입력하여 기본값이 아닌 포트 `8090`을 사용하도록 URL을 변경합니다. 이 방법도 동작하지만, 접근을 차단할 수 있는 포트 보호 메커니즘, 기본 서버 설정과의 불완전한 통합으로 인한 기능 제한, 원시 포트를 사용자에게 노출하는 데 따른 잠재적 보안 위험 등 몇 가지 제약이 있는 최소한의 인터페이스만 얻게 됩니다.
- **리버스 프록시 접근**: 다른 포트가 리버스 프록시되어 `http://<...>.courses.nvidia.com/8090`으로 매핑됩니다(애플리케이션 코드가 바뀌어야 하므로 여기서는 `8091`을 사용합니다). 리버스 프록시는 원시 포트를 사용자로부터 숨겨 백엔드 서비스의 직접 노출을 줄임으로써 보안을 강화합니다. URL 구조도 단순해져서 사용자가 특정 포트 번호를 기억하지 않아도 서비스에 접근할 수 있습니다. 또한 더 나은 로드 밸런싱과 손쉬운 SSL 인증서 관리가 가능해져 더 매끄럽고 안전한 사용자 경험을 제공합니다. 자세한 내용은 코스 범위를 벗어나지만, 관심이 있다면 `composer/deploy/nginx.conf`와 [**`frontend/frontend_server_rproxy.py`**](frontend/frontend_server_rproxy.py)를 살펴보세요.

**아래 셀을 실행하면 링크가 생성됩니다:**

In [ ]:
# %%js
# // Manual Access Without NGINX
# var url = 'http://'+window.location.host+':8090';
# element.innerHTML = '<a style="color:green;" target="_blank" href='+url+'><h1>< Link To Gradio Frontend ></h1></a>';

***시도해 보기 전에 알아 두세요. 프론트엔드 마이크로서비스는 아직... 실제로 동작하지 않습니다. 코스 전반에 걸쳐 이 서비스와 상호작용하고 기능을 활성화할 기회가 여러 번 있을 테니, 해당 부분을 진행할 때는 반드시 코스 환경에서 작업하세요.***

-----

<br>

## **Part 5:** 마무리

이 서비스들의 상태를 확인했다면 첫 번째 노트북은 끝입니다!

### <font color="#76b900">**수고하셨습니다!**</font>

### **다음 단계:**
1. **[선택]** `composer`, 즉 이 마이크로서비스 오케스트레이션 전체와 관련된 배포 전략을 살펴보세요.
2. **[선택]** docker-router 마이크로서비스를 살펴보고 열린 연결 경로가 어떻게 지정되었는지 확인해 보세요.
3. **[선택]** 노트북 상단의 **"생각해 볼 질문" 섹션**을 다시 읽고 가능한 답을 생각해 보세요.

<br>

---

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>